<cell_type>markdown</cell_type># 高级模型服务系统教程 (Advanced Serving Tutorial)

> **前置知识**: FastAPI、负载均衡、异步编程
>
> **学习目标**: 掌握生产环境中的高级服务技术

---

## 为什么需要高级服务技术？

```
┌─────────────────────────────────────────────────────────────┐
│                    生产环境挑战                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  基础服务                        生产环境需求               │
│  ┌─────────────────┐            ┌─────────────────┐        │
│  │ 单点部署        │            │ 高可用部署      │        │
│  │ 无故障处理      │     →      │ 熔断降级        │        │
│  │ 无流量控制      │            │ 速率限制        │        │
│  │ 无可观测性      │            │ 指标追踪        │        │
│  │ 无安全防护      │            │ 输入验证        │        │
│  └─────────────────┘            └─────────────────┘        │
│                                                             │
│  本教程涵盖的高级技术:                                      │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 熔断器模式 - 防止级联故障，快速失败             │   │
│  │  2. 重试策略 - 处理临时故障，指数退避               │   │
│  │  3. 速率限制 - 保护服务，防止过载                   │   │
│  │  4. 指标收集 - 可观测性，性能监控                   │   │
│  │  5. 输入验证 - 安全防护，数据清洗                   │   │
│  │  6. 弹性服务 - 整合所有技术的完整实现               │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **熔断器模式** - 状态机实现，防止级联故障
2. **重试策略** - 指数退避，处理临时故障
3. **速率限制** - 令牌桶和滑动窗口算法
4. **指标收集** - 计数器、直方图、仪表盘
5. **输入验证** - 数据验证和清洗
6. **弹性服务** - 完整的生产级实现

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import asyncio
import time
import random
import json
from enum import Enum
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Callable, Any
from collections import defaultdict, deque
import numpy as np

# 设置随机种子
np.random.seed(42)
random.seed(42)

print("=" * 60)
print("环境准备完成")
print("=" * 60)

print("\n本教程所有组件均为自包含实现，无需额外依赖")
print("可以直接学习和使用各种高级服务技术")

<cell_type>markdown</cell_type>## 1. 熔断器模式 (Circuit Breaker)

**核心概念**: 熔断器用于防止级联故障，当服务出现问题时快速失败

```
┌─────────────────────────────────────────────────────────────┐
│                    熔断器状态机                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  状态转换:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │                                                     │   │
│  │   ┌────────────┐    失败达到阈值    ┌────────────┐ │   │
│  │   │   CLOSED   │ ─────────────────→ │    OPEN    │ │   │
│  │   │   (正常)   │                    │   (熔断)   │ │   │
│  │   └────────────┘                    └─────┬──────┘ │   │
│  │         ↑                                 │        │   │
│  │         │                            超时后        │   │
│  │         │                                 ↓        │   │
│  │         │    成功恢复    ┌────────────────┐        │   │
│  │         └───────────────│   HALF_OPEN    │        │   │
│  │                         │    (半开)      │        │   │
│  │                         └────────┬───────┘        │   │
│  │                                  │                │   │
│  │                             失败则重新 OPEN       │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  状态说明:                                                  │
│  - CLOSED: 正常状态，请求正常通过                          │
│  - OPEN: 熔断状态，请求直接拒绝 (快速失败)                 │
│  - HALF_OPEN: 半开状态，允许少量请求测试恢复               │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 熔断器实现 (自包含实现)
# ============================================================
print("=" * 60)
print("熔断器模式")
print("=" * 60)

class CircuitState(Enum):
    """熔断器状态"""
    CLOSED = "closed"        # 正常状态
    OPEN = "open"            # 熔断状态
    HALF_OPEN = "half_open"  # 半开状态


class CircuitBreaker:
    """
    熔断器
    
    用于防止级联故障，当服务出现问题时快速失败
    
    状态转换:
    - CLOSED → OPEN: 连续失败达到阈值
    - OPEN → HALF_OPEN: 超时后尝试恢复
    - HALF_OPEN → CLOSED: 成功恢复
    - HALF_OPEN → OPEN: 恢复失败
    """
    
    def __init__(self, failure_threshold: int = 5, recovery_timeout: float = 10.0):
        """
        参数:
            failure_threshold: 触发熔断的连续失败次数
            recovery_timeout: 熔断后等待恢复的时间 (秒)
        """
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.last_failure_time = 0.0
        self.success_count = 0
    
    def can_execute(self) -> bool:
        """检查是否可以执行请求"""
        if self.state == CircuitState.CLOSED:
            return True
        
        if self.state == CircuitState.OPEN:
            # 检查是否超时，可以尝试恢复
            if time.time() - self.last_failure_time > self.recovery_timeout:
                self.state = CircuitState.HALF_OPEN
                self.success_count = 0
                print(f"  状态转换: OPEN → HALF_OPEN")
                return True
            return False
        
        # HALF_OPEN 状态允许请求通过
        return True
    
    def record_success(self):
        """记录成功"""
        if self.state == CircuitState.HALF_OPEN:
            self.success_count += 1
            if self.success_count >= 3:
                self.state = CircuitState.CLOSED
                self.failure_count = 0
                self.success_count = 0
                print(f"  状态转换: HALF_OPEN → CLOSED (恢复正常)")
        else:
            self.failure_count = 0
    
    def record_failure(self):
        """记录失败"""
        self.failure_count += 1
        self.last_failure_time = time.time()
        
        if self.state == CircuitState.HALF_OPEN:
            self.state = CircuitState.OPEN
            print(f"  状态转换: HALF_OPEN → OPEN (恢复失败)")
        elif self.failure_count >= self.failure_threshold:
            self.state = CircuitState.OPEN
            print(f"  状态转换: CLOSED → OPEN (连续失败 {self.failure_count} 次)")


# ============================================================
# 测试熔断器
# ============================================================
print("\n熔断器测试:")
print("-" * 40)

cb = CircuitBreaker(failure_threshold=3, recovery_timeout=2)

def unreliable_service():
    """模拟不稳定的服务 (70% 失败率)"""
    if random.random() < 0.7:
        raise Exception("Service failed")
    return "Success"

print("\n模拟 15 个请求:")
for i in range(15):
    if cb.can_execute():
        try:
            result = unreliable_service()
            cb.record_success()
            print(f"  请求 {i+1:2d}: ✓ 成功")
        except:
            cb.record_failure()
            print(f"  请求 {i+1:2d}: ✗ 失败")
    else:
        print(f"  请求 {i+1:2d}: ⊘ 熔断 (快速失败)")
    time.sleep(0.5)

print(f"\n最终状态: {cb.state.value}")

<cell_type>markdown</cell_type>## 2. 重试策略 (Retry Strategy)

**核心概念**: 使用指数退避处理临时故障，避免雪崩效应

```
┌─────────────────────────────────────────────────────────────┐
│                    重试策略对比                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  指数退避 (Exponential Backoff):                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  重试 1: 等待 1s                                    │   │
│  │  重试 2: 等待 2s                                    │   │
│  │  重试 3: 等待 4s                                    │   │
│  │  重试 4: 等待 8s                                    │   │
│  │  ...                                                │   │
│  │  delay = min(base * 2^attempt, max_delay)          │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  线性退避 (Linear Backoff):                                 │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  重试 1: 等待 1s                                    │   │
│  │  重试 2: 等待 2s                                    │   │
│  │  重试 3: 等待 3s                                    │   │
│  │  ...                                                │   │
│  │  delay = min(base * (attempt + 1), max_delay)      │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  抖动 (Jitter):                                             │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  在延迟基础上添加随机抖动，避免多个客户端同时重试   │   │
│  │  jitter = random(0, delay * 0.1)                   │   │
│  │  final_delay = delay + jitter                      │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 重试策略实现 (自包含实现)
# ============================================================
print("=" * 60)
print("重试策略")
print("=" * 60)

class RetryStrategy:
    """
    重试策略
    
    提供多种退避算法，用于处理临时故障
    """
    
    @staticmethod
    def exponential_backoff(attempt: int, base_delay: float = 1.0, max_delay: float = 30.0) -> float:
        """
        指数退避
        
        delay = min(base * 2^attempt, max_delay) + jitter
        
        参数:
            attempt: 当前重试次数 (从 0 开始)
            base_delay: 基础延迟 (秒)
            max_delay: 最大延迟 (秒)
        """
        delay = min(base_delay * (2 ** attempt), max_delay)
        jitter = random.uniform(0, delay * 0.1)  # 添加 10% 抖动
        return delay + jitter
    
    @staticmethod
    def linear_backoff(attempt: int, base_delay: float = 1.0, max_delay: float = 30.0) -> float:
        """
        线性退避
        
        delay = min(base * (attempt + 1), max_delay)
        """
        return min(base_delay * (attempt + 1), max_delay)


def retry_with_backoff(func: Callable, max_retries: int = 3, strategy: str = 'exponential'):
    """
    带退避的重试装饰器
    
    参数:
        func: 要执行的函数
        max_retries: 最大重试次数
        strategy: 退避策略 ('exponential' 或 'linear')
    """
    for attempt in range(max_retries + 1):
        try:
            return func()
        except Exception as e:
            if attempt == max_retries:
                print(f"  重试 {attempt + 1} 次后仍然失败，放弃")
                raise
            
            if strategy == 'exponential':
                delay = RetryStrategy.exponential_backoff(attempt)
            else:
                delay = RetryStrategy.linear_backoff(attempt)
            
            print(f"  尝试 {attempt + 1} 失败，{delay:.2f}s 后重试...")
            time.sleep(delay)


# ============================================================
# 测试重试策略
# ============================================================
print("\n重试策略测试:")
print("-" * 40)

call_count = [0]

def flaky_service():
    """模拟不稳定的服务 (前 2 次失败)"""
    call_count[0] += 1
    if call_count[0] < 3:
        raise Exception("Temporary failure")
    return "Success after retries"

print("\n模拟调用不稳定服务:")
result = retry_with_backoff(flaky_service, max_retries=5)
print(f"\n最终结果: {result}")
print(f"总调用次数: {call_count[0]}")

<cell_type>markdown</cell_type>## 3. 指标收集器 (Metrics Collector)

**核心概念**: 收集服务指标用于监控和告警

```
┌─────────────────────────────────────────────────────────────┐
│                    指标类型                                  │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  计数器 (Counter):                                          │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  只增不减的累计值                                   │   │
│  │  示例: 请求总数、错误总数                          │   │
│  │  requests_total{status="success"} = 1000           │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  直方图 (Histogram):                                        │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  记录值的分布                                       │   │
│  │  示例: 请求延迟分布                                │   │
│  │  latency_seconds{quantile="0.99"} = 0.5            │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  仪表盘 (Gauge):                                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  可增可减的瞬时值                                   │   │
│  │  示例: 当前连接数、内存使用量                      │   │
│  │  active_connections = 42                           │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 指标收集器实现 (自包含实现)
# ============================================================
print("=" * 60)
print("指标收集器")
print("=" * 60)

class MetricsCollector:
    """
    指标收集器
    
    支持三种指标类型:
    - Counter: 计数器，只增不减
    - Histogram: 直方图，记录值的分布
    - Gauge: 仪表盘，可增可减的瞬时值
    """
    
    def __init__(self):
        self.counters = defaultdict(int)
        self.histograms = defaultdict(list)
        self.gauges = {}
    
    def inc_counter(self, name: str, labels: Dict = None, value: int = 1):
        """增加计数器"""
        key = (name, str(labels) if labels else '')
        self.counters[key] += value
    
    def observe_histogram(self, name: str, value: float, labels: Dict = None):
        """记录直方图观测值"""
        key = (name, str(labels) if labels else '')
        self.histograms[key].append(value)
    
    def set_gauge(self, name: str, value: float, labels: Dict = None):
        """设置仪表盘值"""
        key = (name, str(labels) if labels else '')
        self.gauges[key] = value
    
    def get_summary(self) -> Dict[str, Any]:
        """获取指标摘要"""
        summary = {
            'counters': dict(self.counters),
            'gauges': dict(self.gauges),
            'histograms': {}
        }
        
        for key, values in self.histograms.items():
            if values:
                summary['histograms'][str(key)] = {
                    'count': len(values),
                    'mean': np.mean(values),
                    'p50': np.percentile(values, 50),
                    'p95': np.percentile(values, 95),
                    'p99': np.percentile(values, 99),
                }
        
        return summary


# ============================================================
# 测试指标收集器
# ============================================================
print("\n指标收集器测试:")
print("-" * 40)

metrics = MetricsCollector()

# 模拟 100 个请求
for i in range(100):
    latency = random.uniform(0.01, 0.1)
    status = 'success' if random.random() > 0.1 else 'error'
    
    metrics.inc_counter('requests_total', {'status': status})
    metrics.observe_histogram('latency_seconds', latency)

metrics.set_gauge('active_connections', 42)

# 获取摘要
summary = metrics.get_summary()

print("\n计数器:")
for key, value in summary['counters'].items():
    print(f"  {key}: {value}")

print("\n仪表盘:")
for key, value in summary['gauges'].items():
    print(f"  {key}: {value}")

print("\n直方图:")
for key, stats in summary['histograms'].items():
    print(f"  {key}:")
    print(f"    count: {stats['count']}")
    print(f"    mean: {stats['mean']:.4f}s")
    print(f"    p50: {stats['p50']:.4f}s")
    print(f"    p95: {stats['p95']:.4f}s")
    print(f"    p99: {stats['p99']:.4f}s")

<cell_type>markdown</cell_type>## 4. 速率限制器 (Rate Limiter)

**核心概念**: 限制请求速率，保护服务免受过载

```
┌─────────────────────────────────────────────────────────────┐
│                    速率限制算法对比                          │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  令牌桶 (Token Bucket):                                     │
│  ┌─────────────────────────────────────────────────────┐   │
│  │     令牌以固定速率生成                              │   │
│  │            ↓                                        │   │
│  │     ┌─────────────────┐                            │   │
│  │     │  ○ ○ ○ ○ ○ ○    │  ← 桶容量限制             │   │
│  │     └────────┬────────┘                            │   │
│  │              ↓                                      │   │
│  │         请求消耗令牌                                │   │
│  │                                                     │   │
│  │  优点: 允许突发流量 (桶容量)                       │   │
│  │  参数: rate (令牌/秒), capacity (桶容量)           │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  滑动窗口 (Sliding Window):                                 │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  时间窗口: [now - window_size, now]                │   │
│  │                                                     │   │
│  │  ──────────────────────────────────────→ 时间      │   │
│  │       │←── 1秒窗口 ──→│                            │   │
│  │       │  ○ ○ ○ ○ ○    │                            │   │
│  │       │  (5个请求)    │                            │   │
│  │                                                     │   │
│  │  优点: 精确控制，无突发                            │   │
│  │  参数: requests_per_second                         │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 速率限制器实现 (自包含实现)
# ============================================================
print("=" * 60)
print("速率限制器")
print("=" * 60)

class SlidingWindowRateLimiter:
    """
    滑动窗口速率限制器
    
    精确控制每秒请求数，不允许突发
    """
    
    def __init__(self, requests_per_second: int = 10):
        """
        参数:
            requests_per_second: 每秒允许的请求数
        """
        self.rate = requests_per_second
        self.window_size = 1.0  # 1 秒窗口
        self.requests = defaultdict(list)  # 每个客户端的请求时间戳
    
    def is_allowed(self, client_id: str) -> bool:
        """检查请求是否允许"""
        now = time.time()
        window_start = now - self.window_size
        
        # 清理过期请求
        self.requests[client_id] = [
            t for t in self.requests[client_id] if t > window_start
        ]
        
        # 检查是否超过限制
        if len(self.requests[client_id]) >= self.rate:
            return False
        
        self.requests[client_id].append(now)
        return True


class TokenBucketRateLimiter:
    """
    令牌桶速率限制器
    
    允许突发流量，适合大多数场景
    """
    
    def __init__(self, rate: float = 10.0, capacity: int = 20):
        """
        参数:
            rate: 令牌生成速率 (令牌/秒)
            capacity: 桶容量 (最大令牌数)
        """
        self.rate = rate
        self.capacity = capacity
        self.tokens = capacity
        self.last_update = time.time()
    
    def is_allowed(self) -> bool:
        """检查请求是否允许"""
        now = time.time()
        elapsed = now - self.last_update
        
        # 补充令牌
        self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
        self.last_update = now
        
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False


# ============================================================
# 测试速率限制器
# ============================================================
print("\n令牌桶限流器测试:")
print("-" * 40)

limiter = TokenBucketRateLimiter(rate=5, capacity=10)

allowed = 0
denied = 0

print("\n快速发送 20 个请求:")
for i in range(20):
    if limiter.is_allowed():
        allowed += 1
        print(f"  请求 {i+1:2d}: ✓ 通过")
    else:
        denied += 1
        print(f"  请求 {i+1:2d}: ✗ 拒绝")

print(f"\n统计: 通过={allowed}, 拒绝={denied}")
print(f"说明: 初始桶容量=10，所以前 10 个请求通过")

<cell_type>markdown</cell_type>## 5. 输入验证与安全 (Input Validation)

**核心概念**: 验证和清洗输入数据，防止安全漏洞和异常

```
┌─────────────────────────────────────────────────────────────┐
│                    输入验证策略                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  验证 (Validation):                                         │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  严格检查，不符合则拒绝                             │   │
│  │                                                     │   │
│  │  检查项:                                            │   │
│  │  - 类型检查: 是否为数组/列表                       │   │
│  │  - 长度检查: 是否在允许范围内                      │   │
│  │  - 值检查: 是否包含 NaN/Inf                        │   │
│  │  - 范围检查: 值是否在合理范围内                    │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  清洗 (Sanitization):                                       │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  尽量修复，保证数据可用                             │   │
│  │                                                     │   │
│  │  处理:                                              │   │
│  │  - NaN → 0.0                                       │   │
│  │  - Inf → max_value                                 │   │
│  │  - 超范围 → clip 到范围内                          │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  选择建议:                                                  │
│  - 外部输入: 严格验证                                      │
│  - 内部数据: 可以清洗                                      │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 输入验证器实现 (自包含实现)
# ============================================================
print("=" * 60)
print("输入验证器")
print("=" * 60)

class InputValidator:
    """
    输入验证器
    
    提供验证和清洗两种模式
    """
    
    @staticmethod
    def validate_numeric_array(
        data,
        min_len: int = 1,
        max_len: int = 10000,
        min_val: float = -1e6,
        max_val: float = 1e6
    ) -> np.ndarray:
        """
        严格验证数值数组
        
        参数:
            data: 输入数据
            min_len: 最小长度
            max_len: 最大长度
            min_val: 最小值
            max_val: 最大值
            
        返回:
            验证通过的 numpy 数组
            
        异常:
            ValueError: 验证失败
        """
        # 类型检查
        if not isinstance(data, (list, np.ndarray)):
            raise ValueError("Data must be a list or array")
        
        arr = np.array(data, dtype=np.float32)
        
        # 长度检查
        if len(arr) < min_len or len(arr) > max_len:
            raise ValueError(f"Data length must be between {min_len} and {max_len}")
        
        # NaN 检查
        if np.any(np.isnan(arr)):
            raise ValueError("Data contains NaN values")
        
        # Inf 检查
        if np.any(np.isinf(arr)):
            raise ValueError("Data contains Inf values")
        
        # 范围检查
        if np.any(arr < min_val) or np.any(arr > max_val):
            raise ValueError(f"Data values must be between {min_val} and {max_val}")
        
        return arr
    
    @staticmethod
    def sanitize(data, min_val: float = -1e6, max_val: float = 1e6) -> np.ndarray:
        """
        清洗数据 (尽量修复)
        
        处理:
        - NaN → 0.0
        - Inf → max_val
        - 超范围 → clip
        """
        arr = np.array(data, dtype=np.float32)
        arr = np.clip(arr, min_val, max_val)
        arr = np.nan_to_num(arr, nan=0.0, posinf=max_val, neginf=min_val)
        return arr


# ============================================================
# 测试输入验证器
# ============================================================
print("\n输入验证器测试:")
print("-" * 40)

validator = InputValidator()

# 测试正常数据
print("\n1. 正常数据验证:")
valid_data = [1.0, 2.0, 3.0]
result = validator.validate_numeric_array(valid_data)
print(f"   输入: {valid_data}")
print(f"   结果: {result}")

# 测试异常数据验证
print("\n2. 异常数据验证 (应该失败):")
bad_data = [1.0, float('nan'), 3.0]
try:
    validator.validate_numeric_array(bad_data)
except ValueError as e:
    print(f"   输入: {bad_data}")
    print(f"   错误: {e}")

# 测试数据清洗
print("\n3. 异常数据清洗:")
bad_data = [1.0, float('nan'), float('inf'), -1e10]
sanitized = validator.sanitize(bad_data)
print(f"   输入: {bad_data}")
print(f"   清洗后: {sanitized}")

<cell_type>markdown</cell_type>## 6. 弹性推理服务 (Resilient Service)

**核心概念**: 整合所有高级技术，构建生产级弹性服务

```
┌─────────────────────────────────────────────────────────────┐
│                    弹性服务架构                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  请求处理流程:                                              │
│  ┌─────────────────────────────────────────────────────┐   │
│  │                                                     │   │
│  │  请求 → [速率限制] → [熔断检查] → [输入验证]       │   │
│  │              │            │            │            │   │
│  │              ↓            ↓            ↓            │   │
│  │           超限?        熔断?        无效?           │   │
│  │              │            │            │            │   │
│  │              ↓            ↓            ↓            │   │
│  │           拒绝         快速失败      拒绝           │   │
│  │                                                     │   │
│  │                         ↓ 通过                      │   │
│  │                                                     │   │
│  │                    [模型推理]                       │   │
│  │                         │                           │   │
│  │              ┌──────────┴──────────┐               │   │
│  │              ↓                     ↓               │   │
│  │           成功                   失败               │   │
│  │              │                     │               │   │
│  │              ↓                     ↓               │   │
│  │        记录成功              记录失败               │   │
│  │        返回结果              触发熔断?              │   │
│  │                                                     │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  组件:                                                      │
│  - CircuitBreaker: 熔断器                                  │
│  - TokenBucketRateLimiter: 速率限制                        │
│  - MetricsCollector: 指标收集                              │
│  - InputValidator: 输入验证                                │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 弹性推理服务实现 (自包含实现)
# ============================================================
print("=" * 60)
print("弹性推理服务")
print("=" * 60)

class ResilientInferenceService:
    """
    弹性推理服务
    
    整合熔断器、速率限制、指标收集、输入验证
    """
    
    def __init__(self, model_fn: Callable):
        """
        参数:
            model_fn: 模型推理函数
        """
        self.model_fn = model_fn
        self.circuit_breaker = CircuitBreaker(failure_threshold=3, recovery_timeout=5.0)
        self.rate_limiter = TokenBucketRateLimiter(rate=10, capacity=20)
        self.metrics = MetricsCollector()
        self.validator = InputValidator()
    
    def predict(self, data: List[float], client_id: str = 'default') -> np.ndarray:
        """
        执行预测
        
        流程:
        1. 速率限制检查
        2. 熔断器检查
        3. 输入验证
        4. 模型推理
        5. 记录指标
        """
        # 1. 速率限制
        if not self.rate_limiter.is_allowed():
            self.metrics.inc_counter('requests_total', {'status': 'rate_limited'})
            raise Exception('Rate limit exceeded')
        
        # 2. 熔断检查
        if not self.circuit_breaker.can_execute():
            self.metrics.inc_counter('requests_total', {'status': 'circuit_open'})
            raise Exception('Circuit breaker open')
        
        start = time.time()
        try:
            # 3. 输入验证
            validated = self.validator.validate_numeric_array(data)
            
            # 4. 模型推理
            result = self.model_fn(validated)
            
            # 5. 记录成功
            self.circuit_breaker.record_success()
            self.metrics.inc_counter('requests_total', {'status': 'success'})
            self.metrics.observe_histogram('latency_seconds', time.time() - start)
            
            return result
            
        except Exception as e:
            # 记录失败
            self.circuit_breaker.record_failure()
            self.metrics.inc_counter('requests_total', {'status': 'error'})
            raise
    
    def get_health(self) -> Dict[str, Any]:
        """获取服务健康状态"""
        return {
            'circuit_breaker_state': self.circuit_breaker.state.value,
            'metrics': self.metrics.get_summary()
        }


# ============================================================
# 测试弹性服务
# ============================================================
print("\n弹性服务测试:")
print("-" * 40)

def mock_model(x):
    """模拟模型 (20% 失败率)"""
    if random.random() < 0.2:
        raise Exception('Model error')
    return x * 2

service = ResilientInferenceService(mock_model)

print("\n模拟 20 个请求:")
success_count = 0
error_count = 0

for i in range(20):
    try:
        result = service.predict([1.0, 2.0, 3.0])
        success_count += 1
        print(f"  请求 {i+1:2d}: ✓ 成功")
    except Exception as e:
        error_count += 1
        print(f"  请求 {i+1:2d}: ✗ {e}")

print(f"\n统计: 成功={success_count}, 失败={error_count}")

# 获取健康状态
health = service.get_health()
print(f"\n服务健康状态:")
print(f"  熔断器状态: {health['circuit_breaker_state']}")
print(f"  请求计数: {health['metrics']['counters']}")

<cell_type>markdown</cell_type>## 总结

本教程介绍了生产环境中的高级服务技术：

### 核心知识点

| 技术 | 用途 | 关键实现 |
|:-----|:-----|:---------|
| 熔断器 | 防止级联故障 | 状态机 (CLOSED→OPEN→HALF_OPEN) |
| 重试策略 | 处理临时故障 | 指数退避 + 抖动 |
| 速率限制 | 保护服务 | 令牌桶 / 滑动窗口 |
| 指标收集 | 可观测性 | Counter / Histogram / Gauge |
| 输入验证 | 安全防护 | 验证 + 清洗 |

### API 速查

```python
# 熔断器
breaker = CircuitBreaker(failure_threshold=5, recovery_timeout=30.0)
if breaker.can_execute():
    try:
        result = do_request()
        breaker.record_success()
    except Exception:
        breaker.record_failure()

# 重试策略
result = retry_with_backoff(func, max_retries=3, strategy='exponential')

# 速率限制
limiter = TokenBucketRateLimiter(rate=10, capacity=20)
if limiter.is_allowed():
    process_request()

# 指标收集
metrics = MetricsCollector()
metrics.inc_counter('requests_total', {'status': 'success'})
metrics.observe_histogram('latency_seconds', latency)
metrics.set_gauge('active_connections', 42)
summary = metrics.get_summary()

# 输入验证
validator = InputValidator()
validated = validator.validate_numeric_array(data)  # 严格验证
sanitized = validator.sanitize(data)  # 清洗

# 弹性服务
service = ResilientInferenceService(model_fn)
result = service.predict(data)
health = service.get_health()
```

### 生产环境检查清单

```
部署前检查:
✓ 启用熔断器防止级联故障
✓ 配置重试策略处理临时故障
✓ 设置速率限制保护服务
✓ 实现指标收集用于监控
✓ 添加输入验证防止异常数据
✓ 整合所有组件构建弹性服务

监控指标:
✓ 请求成功率 > 99%
✓ P99 延迟 < SLA 要求
✓ 熔断触发次数
✓ 限流拒绝次数
✓ 输入验证失败次数
```

### 技术选型建议

| 场景 | 推荐技术 |
|:-----|:---------|
| 防止雪崩 | 熔断器 + 重试 |
| 保护后端 | 速率限制 |
| 性能监控 | 指标收集 |
| 安全防护 | 输入验证 |
| 生产部署 | 弹性服务 (整合所有) |